<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Assignment_3/MSE1003_Assignment3_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Comment out the below cells when running in Colab

In [ ]:
!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/

In [ ]:
import os
repo = "MSE1003H_RijaAnsari"
print(os.listdir(repo))

In [ ]:
%cd MSE1003H_RijaAnsari/Assignment_3

# Assignment 3: Active Learning for Single-Objective Optimization

## Introduction

### Overview
In this assignment, the goal is to design an active learning strategy that efficiently discovers a red–yellow–blue (RYB) mixture whose 8-channel optical response matches a given target response measured on the Opentron-2 (OT-2) platform. Each experiment corresponds to a choice of dye volumes (R, Y, B) with a fixed total volume of 300 µL, and the OT-2 returns an 8-dimensional intensity vector across different wavelength channels.

The input is as follows:

$x = (R, Y, B), \quad R + Y + B = 300\ \mu\text{L}$

and the corresponding 8-channel output is:


$f(x) = (ch583, ch670, ch510, ch410, ch620, ch470, ch550, ch440).$


The target output for this assignment is:

$y^{\text{target}} = (6794, 7313, 4227, 554, 9325, 2635, 5758, 2061)$

### Single-objective optimization

To make this a single-objective optimization, the scalar objective is defined as the difference between OT-2 output response and the target response. The Euclidean distance is used after normalization: 

$J(x) = \left\|\tilde{f}(x) - \tilde{y}^{\text{target}}\right\|_2^2,$


where $\tilde{f}(x)$ and $\tilde{y}^{\text{target}}$ are the channel-wise normalized response and target, respectively. 

The optimization task is then simply:

$\min_{x \in \mathcal{X}} J(x), \quad \text{subject to } R + Y + B = 300,\ R, Y, B \ge 10$



Because direct evaluation of $J(x)$ requires running an experiment on the OT-2, we will train a surrogate model to approximate the mapping $x \mapsto J(x)$ using existing data (from Assignment 2 and additional student data). 



## Analysis

### Set up environment and assignment folder

Create a new assignment folder and navigate to it
- cd /Users/rija/MSE1003H_RijaAnsari/Assignment_3

Create a virtual environment in the terminal 
- python -m venv .venv  

Create a new text file with the name ".gitignore"
- add the text venv/,pycache/ and .env (if used)

Issues arrived from multiple python versions that kept conflicting with each other  

**Before beginning ensure that Python 3.13.9 is running**

Check everything is in order:
- Move back to the main repo
    - cd /Users/rija/MSE1003H_RijaAnsari
- Make sure this is the main repository url
    - git remote -v
- We need to add our new files from the assignment folder
    - git add . 
    - git commit -m "Assignment 3 structure update"
    - git pull origin main
    - git push origin main

### Data Import and Cleaning

In [ ]:
pip install gpytorch

In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel


import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal

from scipy.stats import norm

random_seed = 1003

In [ ]:
#target definition

target = pd.Series(
    {
        "ch583": 6794,
        "ch670": 7313,
        "ch510": 4227,
        "ch410": 554,
        "ch620": 9325,
        "ch470": 2635,
        "ch550": 5758,
        "ch440": 2061,
    },
    name="target",
)

target


We need to compile previous students data to train the surrogate model. I will be conserving my Assignment 2 data as a test set:

- **Students_data.zip:**  RYB–response pairs collected by other students that will be the primary training data.
- **Background_filtered_data.csv (optional):** Background noise measurements due to ambient light at different positions. When used, these measurements can be subtracted from raw responses to obtain background-corrected intensities.
- **Assignment 2 data:** RYB input and 8-channel output for test.


In [ ]:
#read single csv files

data_dir = Path(os.getcwd()) 

a2_path = data_dir / "color_results.csv"
background_path = data_dir / "background_filtered_data.csv"


# Load what is available; wrap in try/except so the notebook is robust
def safe_read_csv(path, label):
    if path.exists():
        print(f"Loading {label} from {path}")
        return pd.read_csv(path)
    else:
        print(f"WARNING: {label} file not found at {path}. Skipping.")
        return None

a2_df = safe_read_csv(a2_path, "Assignment 2 data")
background_df = safe_read_csv(background_path, "Background data")

a2_df.head() if a2_df is not None else None


In [ ]:
#compile students data into one dataframe

students_folder = Path(data_dir) / "students_data"  

# Find all CSV files in the folder
csv_files = sorted(students_folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {students_folder}")

print(f"Found {len(csv_files)} CSV files in {students_folder}")


compiled_dfs = []

for file in csv_files:
    try:
        df_i = pd.read_csv(file)
        df_i["source"] = file.stem  # tag each row with the filename (without extension)
        compiled_dfs.append(df_i)
        print(f"Loaded {file.name} — shape {df_i.shape}")
    except Exception as e:
        print(f"⚠️ Skipping {file.name} due to read error: {e}")

# Concatenate all loaded CSVs
students_df = pd.concat(compiled_dfs, ignore_index=True)

print("\nFinal merged dataframe shape:", students_df.shape)
students_df.head()


In [ ]:
# Convert stringified dicts into real dicts
students_df["Command"] = students_df["Command"].apply(ast.literal_eval)
students_df["Sensor Data"] = students_df["Sensor Data"].apply(ast.literal_eval)


In [ ]:
# Expand Command (RYB volumes)
cmd_expanded = pd.json_normalize(students_df["Command"])
cmd_expanded = cmd_expanded.rename(columns={"R": "R", "Y": "Y", "B": "B"})

# Expand Sensor Data (8 channels)
sensor_expanded = pd.json_normalize(students_df["Sensor Data"])


In [ ]:
flat_df = pd.concat(
    [
        cmd_expanded,          # R, Y, B
        sensor_expanded,       # ch583 ... ch440
        students_df[["Experiment ID", "timestamp", "source"]]
    ],
    axis=1
)

print("Flattened dataframe shape:", flat_df.shape)
flat_df.head()


In [ ]:
# Identify channel columns automatically
channel_cols = [c for c in flat_df.columns if c.startswith("ch")]

clean_df = flat_df[["R", "Y", "B"] + channel_cols].copy()

print("Clean dataframe shape:", clean_df.shape)
clean_df.head()


### Exploratory Data Analysis

Before training the surrogate model, it is important to understand:

- The coverage of the RYB design space  
- The distribution of raw channel intensities  
- Correlations between channels  
- Whether the dataset contains clusters, gaps, or outliers  

Let's split the data before we get started so we don't have any leakage

In [ ]:
"""#train test split of dataframe

df = clean_df.copy()

#X is the volumes of RYB and Y is the 8 channels
X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

print("Training set size:", X_train.shape, y_train.shape)
print("Test set size:", X_test.shape, y_test.shape)"""

print("Results from simple train/test split are saved in 'simple_split_results.csv'")

In [ ]:
df = clean_df.copy()

# X is the volumes of RYB and y is the 8 channels
X = df[["R", "Y", "B"]]                      # <-- DataFrame
y = df[[f"{ch}" for ch in channel_cols]]     # <-- DataFrame

# 1. Bin each dimension into quantiles (3 bins to avoid sparse strata)
R_bins = pd.qcut(X["R"], q=3, labels=False, duplicates='drop')
Y_bins = pd.qcut(X["Y"], q=3, labels=False, duplicates='drop')
B_bins = pd.qcut(X["B"], q=3, labels=False, duplicates='drop')

# 2. Combine bins into a single stratification label
strata = R_bins * 100 + Y_bins * 10 + B_bins

# 3. Perform a stratified split
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_idx, val_idx in splitter.split(X, strata):
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()


In [ ]:
#ternary plot 
fig = px.scatter_ternary(
    X_train, 
    a="R", 
    b="Y", 
    c="B",
)

fig.update_traces(marker=dict(color='black', size=5))

fig.update_layout(
    ternary={
        'sum': 100,
        "aaxis": {
            "title": {"text": "Red", "font": {"color": "red", "size": 20}},
            "tickfont": {"color": "red"},
            "linecolor": "red"
        },
        "baxis": {
            "title": {"text": "Yellow", "font": {"color": "yellow", "size": 20}},
            "tickfont": {"color": "black"},
            "linecolor": "yellow"
        },
        "caxis": {
            "title": {"text": "Blue", "font": {"color": "blue", "size": 20}},
            "tickfont": {"color": "blue"},
            "linecolor": "blue"
        }
    }
)
fig.show()

We see a good coverage of our search space from out train data. This should suffice as input for our surrogate model.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(8,8))
axes = axes.flatten()

for ax, ch in zip(axes, channel_cols):
    sns.histplot(df[ch], kde=True, ax=ax)
    ax.set_title(f"Distribution of {ch}")

plt.tight_layout()
plt.show()


In [ ]:
subset = [f"{ch}" for ch in channel_cols]

sns.pairplot(df[subset], corner=True, diag_kind="kde")
plt.suptitle("Pairwise Relationships Between Channels", y=1.02)
plt.show()


Strong positive correlations: Several channel pairs form clear upward‑sloping clusters, indicating a proportional relationship between channel intensities. 

Smooth, continuous structure: Smooth scatterplots are ideal for Gaussian Process Regression with no extreme outliers.

In [ ]:
plt.figure(figsize=(6, 4))
corr = df[[f"{ch}" for ch in channel_cols]].corr()

sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap of Channels")
plt.tight_layout()
plt.show()


### Surrogate Model

In [ ]:
# Separate scalers
x_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fit on training data ONLY
X_train_scaled = x_scaler.fit_transform(X_train)
#X_test_scaled  = x_scaler.transform(X_test)
X_val_scaled   = x_scaler.transform(X_val)

y_train_scaled = y_scaler.fit_transform(y_train)
#y_test_scaled  = y_scaler.transform(y_test)
y_val_scaled  = y_scaler.transform(y_val)


Normalization of the outputs is important as the eight optical channels measured by the OT‑2 span very different numerical ranges.  
For example, `ch410` may produce values in the hundreds, while `ch620` or `ch670` can reach values above 9000. Normalization prevents dominance of high-magnitude channels and enables stability of the surrogate model. It is important to note that the target response must also be normalzied with the same training statistics for proper comparison. 


We will primarily use Gaussian Process Regression (GPR) as the surrogate model due to its suitability for small datasets and its ability to provide calibrated uncertainty estimates, which are crucial for active learning.

Using this surrogate, we will implement a sequential acquisition policy (e.g., Expected Improvement) to select up to 26 new RYB combinations to query on the OT-2. At each iteration, the surrogate will be updated with the new data, and we will track:

- Convergence of the predicted response to the target response.
- Improvement in predictive accuracy of the surrogate model.
- Evolution of predictive uncertainty and sample selection in the RYB space.

The following sections describe the data preparation, surrogate modeling, and active learning strategy in detail.


In [ ]:
results = pd.DataFrame(columns=['mae_train', 
                                'rmse_train', 
                                #'maxe_train', 
                                #'mape_train', 
                                'r2_train', 
                                #'mae_val', 
                                #'rmse_val',
                                # 'r2_val',
                                #'maxe_val', 
                                #'mape_val' 
                                ])

def compute_metrics(y_true, y_pred, model, set_type):
    mae = mean_absolute_error(y_true, y_pred)
    #mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    #maxe = max_error(y_true, y_pred)
    #mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics_dict = {
    f"mae_{set_type}":  [mae],
    f"rmse_{set_type}":  [rmse],
   # f"maxe_{set_type}": [maxe],
   # f"mape_{set_type}": [mape],
    f"r2_{set_type}":   [r2]}

    for col, val in metrics_dict.items():
            results.loc[model, col] = val

    return results.loc[model]

In [ ]:
kernel = RBF()
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results with RF_opt
results.loc['RBF'] = [0] * len(results.columns)
y_pred_train_RBF = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF = gpr_model.predict(X_test_scaled)
y_pred_val_RBF = gpr_model.predict(X_val_scaled)


results.loc['RBF'] = compute_metrics(y_train_scaled, y_pred_train_RBF, "RBF", "train")
#results.loc['RBF'] = compute_metrics(y_test_scaled, y_pred_test_RBF, "RBF", "test")
results.loc['RBF'] = compute_metrics(y_val_scaled, y_pred_val_RBF, "RBF", "val")

results

Radial basis function on its own works well on the trained data but poorly on the validation set.

Let's try adding white kernel to see what happens

In [ ]:
kernel = RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0) 
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_WK'] = [0] * len(results.columns)
y_pred_train_RBF_WK = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF_WK = gpr_model.predict(X_test_scaled)
y_pred_val_RBF_WK = gpr_model.predict(X_val_scaled)


results.loc['RBF_WK'] = compute_metrics(y_train_scaled, y_pred_train_RBF_WK, "RBF_WK", "train")
#results.loc['RBF_WK'] = compute_metrics(y_test_scaled, y_pred_test_RBF_WK, "RBF_WK", "test")
results.loc['RBF_WK'] = compute_metrics(y_val_scaled, y_pred_val_RBF_WK, "RBF_WK", "val")

results

Does not like white kernel, let's try matern

In [ ]:
kernel = RBF(length_scale=1.0) + Matern(length_scale=1.0, nu=1.5)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_MT'] = [0] * len(results.columns)
y_pred_train_RBF_MT = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF_MT = gpr_model.predict(X_test_scaled)
y_pred_val_RBF_MT = gpr_model.predict(X_val_scaled)

results.loc['RBF_MT'] = compute_metrics(y_train_scaled, y_pred_train_RBF_MT, "RBF_MT", "train")
#results.loc['RBF_MT'] = compute_metrics(y_test_scaled, y_pred_test_RBF_MT, "RBF_MT", "test")
results.loc['RBF_MT'] = compute_metrics(y_val_scaled, y_pred_val_RBF_MT, "RBF_MT", "val")

results

Matern is showing severe overfitting, let's try rational quadratic

In [ ]:
kernel = RBF(length_scale=1.0) + RationalQuadratic(length_scale=1.0, alpha=1.0)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_RQ'] = [0] * len(results.columns)
y_pred_train_RBF_RQ = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF_RQ = gpr_model.predict(X_test_scaled)
y_pred_val_RBF_RQ = gpr_model.predict(X_val_scaled)

results.loc['RBF_RQ'] = compute_metrics(y_train_scaled, y_pred_train_RBF_RQ, "RBF_RQ", "train")
#results.loc['RBF_RQ'] = compute_metrics(y_test_scaled, y_pred_test_RBF_RQ, "RBF_RQ", "test")
results.loc['RBF_RQ'] = compute_metrics(y_val_scaled, y_pred_val_RBF_RQ, "RBF_RQ", "val")

results

Better but just about as good as RBF on its own.

In [ ]:
kernel = RBF(length_scale=1.0) + ConstantKernel(constant_value=1.0)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_CK'] = [0] * len(results.columns)
y_pred_train_RBF_CK = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF_CK = gpr_model.predict(X_test_scaled)
y_pred_val_RBF_CK = gpr_model.predict(X_val_scaled)

results.loc['RBF_CK'] = compute_metrics(y_train_scaled, y_pred_train_RBF_CK, "RBF_CK", "train")
#results.loc['RBF_CK'] = compute_metrics(y_test_scaled, y_pred_test_RBF_CK, "RBF_CK", "test")
results.loc['RBF_CK'] = compute_metrics(y_val_scaled, y_pred_val_RBF_CK, "RBF_CK", "val")
results

In [ ]:
kernel = 1.0 * RBF(length_scale=np.ones(3), length_scale_bounds=(1e-2, 1e3)) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_WK2'] = [0] * len(results.columns)
y_pred_train_RBF_WK2 = gpr_model.predict(X_train_scaled)
#y_pred_test_RBF_WK2 = gpr_model.predict(X_test_scaled)
y_pred_val_RBF_WK2 = gpr_model.predict(X_val_scaled)

results.loc['RBF_WK2'] = compute_metrics(y_train_scaled, y_pred_train_RBF_WK2, "RBF_WK2", "train")
#results.loc['RBF_WK2'] = compute_metrics(y_test_scaled, y_pred_test_RBF_WK2, "RBF_WK2", "test")
results.loc['RBF_WK2'] = compute_metrics(y_val_scaled, y_pred_val_RBF_WK2, "RBF_WK2", "val")
results

In [ ]:
kernel = RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e3)) #+ WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr = GPR(kernel=kernel, normalize_y=False,random_state=random_seed)
multi_gpr = MultiOutputRegressor(gpr)
multi_gpr.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['MOR'] = [0] * len(results.columns)
y_pred_train_MOR = multi_gpr.predict(X_train_scaled)
#y_pred_test_MOR = multi_gpr.predict(X_test_scaled)
y_pred_val_MOR = multi_gpr.predict(X_val_scaled)

results.loc['MOR'] = compute_metrics(y_train_scaled, y_pred_train_MOR, "MOR", "train")
#results.loc['MOR'] = compute_metrics(y_test_scaled, y_pred_test_MOR, "MOR", "test")
results.loc['MOR'] = compute_metrics(y_val_scaled, y_pred_val_MOR, "MOR", "val")
results

In [ ]:
kernel = RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e3)) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr = GPR(kernel=kernel, normalize_y=False,random_state=random_seed)
multi_gpr = MultiOutputRegressor(gpr)
multi_gpr.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['MOR_WK'] = [0] * len(results.columns)
y_pred_train_MOR_WK = multi_gpr.predict(X_train_scaled)
#y_pred_test_MOR_WK = multi_gpr.predict(X_test_scaled)
y_pred_val_MOR_WK = multi_gpr.predict(X_val_scaled)

results.loc['MOR_WK'] = compute_metrics(y_train_scaled, y_pred_train_MOR_WK, "MOR_WK", "train")
#results.loc['MOR_WK'] = compute_metrics(y_test_scaled, y_pred_test_MOR_WK, "MOR_WK", "test")
results.loc['MOR_WK'] = compute_metrics(y_val_scaled, y_pred_val_MOR_WK, "MOR_WK", "val")
results

In [ ]:
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks):
        super().__init__(train_x, train_y, likelihood)

        # Multitask mean: wraps a base mean and expands to (N, num_tasks)
        self.mean_module = MultitaskMean(
            base_means=ConstantMean(),
            num_tasks=num_tasks
        )

        # Base kernel over inputs
        data_kernel = ScaleKernel(RBFKernel(ard_num_dims=train_x.shape[-1]))

        # ICM multitask kernel
        self.covar_module = MultitaskKernel(
            data_covar_module=data_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)      # shape: (N, num_tasks)
        covar_x = self.covar_module(x)    # multitask covariance
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)


In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]


In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
num_tasks = y_train_t.shape[1]

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskGPModel(X_train_t, y_train_t, likelihood, num_tasks)

model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(300):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = -mll(output, y_train_t)
    loss.backward()
    optimizer.step()


In [ ]:
model.eval()
likelihood.eval()

#X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    preds_train = likelihood(model(X_train_t)) 
    #preds_test = likelihood(model(X_test_t))
    preds_val = likelihood(model(X_val_t)) 

y_pred_train_gpy = preds_train.mean.numpy() 
#y_pred_test_gpy = preds_test.mean.numpy()
y_pred_val_gpy = preds_val.mean.numpy()


In [ ]:
results.loc['GPyTorch'] = [0] * len(results.columns)

results.loc['GPyTorch'] = compute_metrics(y_train_scaled, y_pred_train_gpy, "GPyTorch", "train")
#results.loc['GPyTorch'] = compute_metrics(y_test_scaled, y_pred_test_gpy, "GPyTorch", "test")
results.loc['GPyTorch'] = compute_metrics(y_val_scaled,  y_pred_val_gpy,"GPyTorch", "val")

results

Note: Sometimes the cell below gets stuck - just rerun 

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskGPModel(X_train_t, y_train_t, likelihood, num_tasks)

model.train()
likelihood.train()

# Improvement 1: Higher LR
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# Improvement 2: More iterations
num_iters = 1000

# Improvement 3: Jitter for stability
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(num_iters):
    optimizer.zero_grad()

    with gpytorch.settings.cholesky_jitter(1e-3):
        output = model(X_train_t)
        loss = -mll(output, y_train_t)

    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Iter {i} - Loss: {loss.item():.4f}")


In [ ]:
model.eval()
likelihood.eval()

#X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    preds_train = likelihood(model(X_train_t))
    #preds_test  = likelihood(model(X_test_t))
    preds_val   = likelihood(model(X_val_t))
    

y_pred_train_gpy_mod = preds_train.mean.numpy()
#y_pred_test_gpy_mod  = preds_test.mean.numpy()
y_pred_val_gpy_mod   = preds_val.mean.numpy()



In [ ]:
results.loc['GPyTorch_mod'] = [0] * len(results.columns)

results.loc['GPyTorch_mod'] = compute_metrics(y_train_scaled, y_pred_train_gpy_mod,"GPyTorch","train")
#results.loc['GPyTorch_mod'] = compute_metrics(y_test_scaled, y_pred_test_gpy_mod,"GPyTorch","test")
results.loc['GPyTorch_mod'] = compute_metrics(y_val_scaled,y_pred_val_gpy_mod,"GPyTorch","val")
results

In [ ]:
from gpytorch.kernels import SpectralMixtureKernel

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
#X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32)
X_val_t   = torch.tensor(X_val_scaled,   dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]
input_dim = X_train_t.shape[-1]

# ----------------------------
class MultitaskSMGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks, input_dim):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = MultitaskMean(
            base_means=ConstantMean(),
            num_tasks=num_tasks
        )

        sm_kernel = SpectralMixtureKernel(
            num_mixtures=4,
            ard_num_dims=input_dim
        )

        self.covar_module = MultitaskKernel(
            data_covar_module=sm_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskSMGPModel(X_train_t, y_train_t, likelihood, num_tasks, input_dim)


model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 800
for i in range(num_iters):
    optimizer.zero_grad()
    with gpytorch.settings.cholesky_jitter(1e-3):
        output = model(X_train_t)
        loss = -mll(output, y_train_t)
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"Iter {i} - Loss: {loss.item():.4f}")

###
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    preds_train = likelihood(model(X_train_t))
    #preds_test  = likelihood(model(X_test_t))
    preds_val   = likelihood(model(X_val_t))

y_pred_train_gpy_mod2 = preds_train.mean.numpy()
#y_pred_test_gpy_mod2  = preds_test.mean.numpy()
y_pred_val_gpy_mod2   = preds_val.mean.numpy()


In [ ]:
results.loc['GPyTorch_mod2'] = [0] * len(results.columns)

results.loc['GPyTorch_mod2'] = compute_metrics(y_train_scaled,y_pred_train_gpy_mod2,"GPyTorch_mod2","train")
#results.loc['GPyTorch_mod2'] = compute_metrics(y_test_scaled, y_pred_test_gpy_mod2,"GPyTorch_mod2","test")
results.loc['GPyTorch_mod2'] = compute_metrics(y_val_scaled, y_pred_val_gpy_mod2,"GPyTorch_mod2","val")

results


In [ ]:
#save results table to csv with name simple split
#results.to_csv("simple_split_results.csv", index=True)

'''read results table from csv
simple_split = pd.read_csv("simple_split_results.csv", index_col=0)
simple_split'''

In [ ]:
#make parity plot train and test for best model (GPyTorch_mod)
plt.figure(figsize=(6,6))
plt.scatter(y_train_scaled.flatten(), y_pred_train_gpy_mod2.flatten(), alpha=0.5, label="Train", color="blue")
#plt.scatter(y_test_scaled.flatten(), y_pred_test_gpy_mod2.flatten(), alpha=0.5, label="Test", color="orange")
plt.scatter(y_val_scaled.flatten(), y_pred_val_gpy_mod2.flatten(), alpha=0.5, label="Val", color="green")
plt.plot([y_train_scaled.min(), y_train_scaled.max()], [y_train_scaled.min(), y_train_scaled.max()], 'k--', label="Ideal")
plt.xlabel("True Values (scaled)")
plt.ylabel("Predicted Values (scaled)") 
plt.title("Parity Plot for GPyTorch_mod2")
plt.legend()
plt.tight_layout()
plt.show()

There is a big cluster of values in the validation set that just collapse to zero, which is the cause for the low validation metrics. 

### Acquisition Policy 

In [ ]:
def gp_predict_mean_std(model, likelihood, X_scaled):
    
    #Predicts 8 channels for each input.
    #Returns mean and std of shape (N, 8).
    
    model.eval()
    likelihood.eval()
    X_t = torch.tensor(X_scaled, dtype=torch.float32)
    
    with torch.no_grad():
        preds = likelihood(model(X_t))
        # This returns (N, 8) for multitask models
        mean = preds.mean.numpy()
        std = preds.stddev.numpy()
        
    return mean, std

def acquisition_EI(model, likelihood, X_candidates_scaled, best_y, target_scaled):
    """
    Calculates EI by converting 8-channel predictions into a scalar distance.
    """
    # 1. Get 8-channel predictions (N, 8)
    mu_channels, sigma_channels = gp_predict_mean_std(model, likelihood, X_candidates_scaled)
    
    # 2. Calculate Predicted Distance (the mean of our objective)
    # J = sum((channel_prediction - target)^2)
    mu_dist = np.sum((mu_channels - target_scaled)**2, axis=1)
    
    # 3. Calculate an aggregate uncertainty (sigma) for the distance
    # We use the average standard deviation across the 8 channels as a proxy
    sigma_dist = np.mean(sigma_channels, axis=1)
    sigma_dist = np.maximum(sigma_dist, 1e-9)
    
    # 4. Standard EI calculation for MINIMIZATION
    # best_y MUST be the lowest distance found so far
    Z = (best_y - mu_dist) / sigma_dist
    ei = (best_y - mu_dist) * norm.cdf(Z) + sigma_dist * norm.pdf(Z)
    
    return ei


def propose_next_batch(model, likelihood, X_pool_scaled, X_pool_raw, best_y, target_scaled, n_points=5):
    """
    Proposes 5 points using the distance-based EI.
    """
    ei_values = acquisition_EI(model, likelihood, X_pool_scaled, best_y, target_scaled)
    
    sorted_indices = np.argsort(ei_values)[::-1]
    best_indices = sorted_indices[:n_points]
    
    next_batch_scaled = X_pool_scaled[best_indices]
    
    # CHANGE THIS LINE: Remove .iloc because X_pool_raw is a NumPy array
    next_batch_raw = X_pool_raw[best_indices] 
    
    return next_batch_raw, next_batch_scaled, ei_values[best_indices]

In [ ]:
def generate_candidate_pool(step=5):
    """Generates valid (R, Y, B) points where sum=300 and each >= 10."""
    candidates = []
    for r in range(10, 301, step):
        for y in range(10, 301 - r + 1, step):
            b = 300 - r - y
            if b >= 10:
                candidates.append([float(r), float(y), float(b)])
    return np.array(candidates)

# 1. Calculate the distance for your current training data
# Ensure target_scaled is defined as the 8-channel target normalized

#target_raw = np.array([6794, 7313, 4227, 554, 9325, 2635, 5758, 2061])
#target_scaled = y_scaler.transform(target_raw).reshape(1, -1).flatten()

#y_train_objective = np.sum((y_train_scaled - target_scaled)**2, axis=1)
#best_y_current = np.min(y_train_objective)

# 1. Define the raw target values
target_raw_values = [6794, 7313, 4227, 554, 9325, 2635, 5758, 2061]

# 2. Convert to a DataFrame using the same column names as your training data
# This fixes the Shape error AND the Name warning
target_df = pd.DataFrame([target_raw_values], columns=y_train.columns)

# 3. Transform and flatten into a 1D vector for use in your distance calculations
target_scaled = y_scaler.transform(target_df).flatten()

# 4. Calculate the distance for your current training data
# y_train_scaled is (N, 8), target_scaled is (8,)
y_train_objective = np.sum((y_train_scaled - target_scaled)**2, axis=1)
best_y_current = np.min(y_train_objective)

print(f"Target Scaled: {target_scaled}")
print(f"Best Distance so far: {best_y_current}")

X_pool_raw = generate_candidate_pool(step=10) # 10mL steps
X_pool_scaled = x_scaler.transform(X_pool_raw)


# 2. Propose the next 5 points
next_batch_raw, next_batch_scaled, eis = propose_next_batch(
    model, 
    likelihood, 
    X_pool_scaled, 
    X_pool_raw, 
    best_y_current, 
    target_scaled, # Pass the target here!
    n_points=5
)

print("Proposed Next Batch:")
print("R Y B (volumes):")
print(next_batch_raw)

### Active Learning Loop

#### Iteration 1: 5 Batch Samples

In [ ]:
new_data_path = data_dir / "Rija_Ansari_5.csv"
new_df_5 = safe_read_csv(new_data_path, "New data from Rija_Ansari_5.csv")
new_df_5

In [ ]:
#clean new_df_5 to have only R, Y, B and channels
new_df_5["Command"] = new_df_5["Command"].apply(ast.literal_eval)
new_df_5["Sensor Data"] = new_df_5["Sensor Data"].apply(ast.literal_eval)
cmd_expanded_5 = pd.json_normalize(new_df_5["Command"])
cmd_expanded_5 = cmd_expanded_5.rename(columns={"R": "R", "Y": "Y", "B": "B"})
sensor_expanded_5 = pd.json_normalize(new_df_5["Sensor Data"])
flat_df_5 = pd.concat(
    [
        cmd_expanded_5,          # R, Y, B
        sensor_expanded_5,       # ch583 ... ch440
    ],
    axis=1
)   
channel_cols_5 = [c for c in flat_df_5.columns if c.startswith("ch")]
clean_df_5 = flat_df_5[["R", "Y", "B"] + channel_cols_5].copy()
clean_df_5

In [ ]:
#calculate distance to target for new data
X_new_5 = clean_df_5[["R", "Y", "B"]]
y_new_5 = clean_df_5[[f"{ch}" for ch in channel_cols_5]]    
X_new_5_scaled = x_scaler.transform(X_new_5)
#y_new_5_scaled = y_scaler.transform(y_new_5)

#ordered columns of y_new_5 to match y_train_scaled
expected_cols = y_scaler.feature_names_in_
y_new_5_aligned = y_new_5[expected_cols]
y_new_5_scaled = y_scaler.transform(y_new_5_aligned)

y_new_5_objective = np.sum((y_new_5_scaled - target_scaled)**2, axis=1)
print(f"Distance to target for new data: {y_new_5_objective}")
#print the closest sample index and its RYB values
closest_idx = np.argmin(y_new_5_objective)
print(f"Closest sample index: {closest_idx}")
print(f"RYB values of closest sample: {X_new_5.iloc[closest_idx].values}")


In [ ]:
# Metric tracking lists
iteration_history = []
dist_to_target_history = []  # Graph 1: Convergence
mae_history = []       # Graph 2a: MAE 
rmse_history = []      # Graph 2b: RMSE
r2_history = []        # Graph 2c: R²
uncertainty_history = []    # Graph 2b: Mean Std Dev
input_samples_history = []  # Graph 3: RYB volumes sampled

In [ ]:
# --- 1. Calculate Distances ---

# Iteration 0: The best (minimum) distance in your original training data
initial_distances = np.sum((y_train_scaled[:-(len(y_new_5_scaled))] - target_scaled)**2, axis=1)
best_dist_it0 = np.min(initial_distances)

# Iteration 1: The distances of the 5 new points you just added
# We use the mean distance of the batch to show the average quality of the proposal
new_points_distances = np.sum((y_new_5_scaled - target_scaled)**2, axis=1)
avg_dist_it1 = np.mean(new_points_distances)
best_dist_it1 = np.min(new_points_distances)

# Store for plotting
iterations = [0, 1]
best_distances = [best_dist_it0, best_dist_it1]
batch_avg_distances = [best_dist_it0, avg_dist_it1] # Start it0 at best for comparison

In [ ]:
# --- 1. Calculate Distances ---
# (Assuming you have y_train_scaled and target_scaled from the previous steps)

# Iteration 0: Best distance in original data
initial_distances = np.sum((y_train_scaled[:-(len(y_new_5_scaled))] - target_scaled)**2, axis=1)
best_dist_it0 = np.min(initial_distances)

# Iteration 1: Distances of the 5 new points
new_points_distances = np.sum((y_new_5_scaled - target_scaled)**2, axis=1)
best_dist_it1 = np.min(new_points_distances)

# Tracking data
iters = [0, 1]
best_dists = [best_dist_it0, best_dist_it1]

# --- 2. Plotting ---
plt.figure(figsize=(9, 6))

# The "Target" line (Distance = 0)
plt.axhline(0, color='black', linestyle='-', linewidth=2, label='Perfect Target (J=0)')
plt.fill_between([0, 1], 0, best_dists, color='green', alpha=0.1, label='Error Gap')

# Convergence line
plt.plot(iters, best_dists, marker='o', markersize=10, color='tab:blue', 
         linewidth=3, label='Best Sampled Distance')

# Plot the average distance of the newly proposed batch
plt.scatter(iterations[1], batch_avg_distances[1], color='tab:orange', 
            s=100, label='Avg. Distance of New Batch', zorder=5)

plt.title("Convergence: Distance to Target Response", fontsize=14)
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Squared Euclidean Distance $J(x)$", fontsize=12)
plt.xticks([0, 1])
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 1. Align the target values with the column order of your new data
# Assuming target_raw_values is a list/array and you have a corresponding list of channel names
original_channel_names = y_train.columns.tolist()  # This should match the order of your target_raw_values

target_df = pd.Series(target_raw_values, index=original_channel_names)
target_aligned = target_df.reindex(y_new_5.columns).values 

# 2. Now use target_aligned instead of target_raw in your plotting code
target_raw = target_aligned 

# 1. Calculate raw distances to identify the best match visually
raw_responses = y_new_5.values 
target_raw = np.array(target_raw)
raw_distances = np.sum((raw_responses - target_raw)**2, axis=1)
best_idx_raw = np.argmin(raw_distances)

# 2. Setup colors and labels
# We'll use a standard color cycle but make the "non-best" ones lighter
colors = ['tab:blue', 'tab:orange', 'tab:purple', 'tab:red', 'tab:brown']
channels = y_new_5.columns.tolist()

plt.figure(figsize=(12, 7))

# --- Plot the RAW Target Response ---
plt.plot(channels, target_raw, 'k--', marker='s', label='TARGET (Ideal)', linewidth=3, zorder=10)

# --- Loop through all 5 samples ---
for i in range(len(raw_responses)):
    current_raw = raw_responses[i]
    current_color = colors[i % len(colors)] # Cycle colors if batch > 5
    
    if i == best_idx_raw:
        # Highlight the best match: Solid color, thick line, high z-order
        plt.plot(channels, current_raw, color='tab:green', marker='o', 
                 linewidth=4, label=f'SAMPLE {i} (BEST MATCH)', zorder=5)
        
        # Fill the error gap specifically for the best one
        plt.fill_between(channels, target_raw, current_raw, color='tab:green', alpha=0.1)
    else:
        # Other samples: Unique colors but lighter (alpha=0.4)
        plt.plot(channels, current_raw, color=current_color, alpha=0.4, 
                 linewidth=1.5, marker='.', label=f'Sample {i}')

# Formatting for clarity
plt.title("Spectral Profile: Comparison of All 5 Batch Samples (Raw)", fontsize=16)
plt.ylabel("Raw Sensor Intensity (Counts)", fontsize=12)
plt.xlabel("Wavelength Channel", fontsize=12)

# Place legend to the right so it doesn't overlap the curves
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, frameon=True)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()

plt.show()

# Print the volumes associated with each index for quick reference
print("Volumes sampled in this batch:")
print(X_new_5)

In [ ]:
# Create a dictionary to store 'snapshots' for comparison
comparison_log = {
    'iteration': [],
    'mae_pre': [],  'mae_post': [],
    'unc_pre': [],  'unc_post': [],
    'dist_pre': [], 'dist_post': []
}

# --- Inside your AL Loop ---

# 1. 'PRE' Metrics (Current model on existing data)
model.eval()
with torch.no_grad():
    preds_pre = likelihood(model(X_train_t))
    comparison_log['mae_pre'].append(mean_absolute_error(y_train_scaled, preds_pre.mean.numpy()))
    comparison_log['unc_pre'].append(np.mean(np.sqrt(preds_pre.variance.numpy())))
    comparison_log['dist_pre'].append(np.min(np.sum((preds_pre.mean.numpy() - target_scaled)**2, axis=1)))


In [ ]:
# --- STEP A: INPUT NEW DATA ---
# Replace these with your actual 5 new results

#make dataframe new_volumes_raw with columns R, Y, B and 5 rows of new volumes from clean_df_5
new_volumes_raw = clean_df_5[["R", "Y", "B"]].copy()

#make dataframe new_responses_raw with columns ch583, ch620, ch470, ch550, ch440 and 5 rows of new responses from clean_df_5
new_responses_raw = clean_df_5[channel_cols_5].copy()
#make sure the columns of new_responses_raw are in the same order as y_train_scaled
new_responses_raw = new_responses_raw[y_train.columns]

# --- STEP B: UPDATE TRAINING SET ---
X_train = pd.concat([X_train, new_volumes_raw], ignore_index=True)
y_train = pd.concat([y_train, new_responses_raw], ignore_index=True)

# --- STEP C: RETRAIN SCALERS AND MODEL ---
X_train_scaled = x_scaler.fit_transform(X_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# Update GPyTorch model with new data
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

model.set_train_data(inputs=X_train_t, targets=y_train_t, strict=False)

# Re-run training (MCMC/Adam) for ~50-100 epochs to adjust to new data
model.train()
likelihood.train()
# ... (insert your optimizer loop here) ...

# --- STEP D: CALCULATE METRICS ---
model.eval()
likelihood.eval()
with torch.no_grad():
    preds = likelihood(model(X_train_t))
    mean = preds.mean.numpy()
    std = torch.sqrt(preds.variance).numpy()

# 1. Convergence: Distance of predicted mean to target_scaled
current_dist = np.mean(np.sum((mean - target_scaled)**2, axis=1))
dist_to_target_history.append(current_dist)

# 2. Accuracy: Mean Absolute Error on training set
current_mae = mean_absolute_error(y_train_scaled, mean)
current_rmse = root_mean_squared_error(y_train_scaled, mean)
current_r2 = r2_score(y_train_scaled, mean) 
mae_history.append(current_mae)
rmse_history.append(current_rmse)
r2_history.append(current_r2)

# 3. Uncertainty: Average standard deviation
current_unc = np.mean(std)
uncertainty_history.append(current_unc)

# 4. Input tracking
input_samples_history.append(new_volumes_raw.values)

print(f"Iteration Complete. Best Distance: {np.min(dist_to_target_history):.4f}")

In [ ]:
# ... [Perform Step A & B: Add new data and retrain model] ...

# 2. 'POST' Metrics (Retrained model on updated data)
model.eval()
with torch.no_grad():
    preds_post = likelihood(model(X_train_t)) # X_train_t now includes the 5 new points
    comparison_log['mae_post'].append(mean_absolute_error(y_train_scaled, preds_post.mean.numpy()))
    comparison_log['unc_post'].append(np.mean(np.sqrt(preds_post.variance.numpy())))
    comparison_log['dist_post'].append(np.min(np.sum((preds_post.mean.numpy() - target_scaled)**2, axis=1)))

In [ ]:
def plot_parity_comparison(y_true, y_pred_pre, y_pred_post):
    fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    
    # Pre-update
    ax[0].scatter(y_true, y_pred_pre, alpha=0.5, color='gray')
    ax[0].plot([0, 1], [0, 1], '--k') # 45-degree line
    ax[0].set_title("Before New Samples")
    
    # Post-update
    ax[1].scatter(y_true, y_pred_post, alpha=0.5, color='blue')
    ax[1].plot([0, 1], [0, 1], '--k')
    ax[1].set_title("After New Samples (Retrained)")
    
    plt.show()

In [ ]:
# Using a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Plot 1: Convergence WITH Target Baseline (Top Left) ---
axes[0, 0].plot(dist_to_target_history, marker='o', color='tab:red', linewidth=2, label='Current Best')
axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=2, label='Target (0)')
axes[0, 0].set_title("1. Convergence (with Target)", fontsize=14)
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Distance $J(x)$")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# --- Plot 2: Convergence WITHOUT Target Baseline (Top Right) ---
# This allows for a better zoom on the actual values
axes[0, 1].plot(dist_to_target_history, marker='o', color='tab:red', linewidth=2)
axes[0, 1].set_title("2. Convergence (Zoomed)", fontsize=14)
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("Distance $J(x)$")
axes[0, 1].grid(True, alpha=0.3)

# --- Plot 3: MAE & Uncertainty Evolution (Bottom Left) ---
ax3_twin = axes[1, 0].twinx()
lns1 = axes[1, 0].plot(mae_history, marker='s', label='MAE', color='tab:blue', linewidth=2)
lns2 = ax3_twin.plot(uncertainty_history, marker='^', label='Uncertainty ($\sigma$)', color='tab:orange', alpha=0.7)

axes[1, 0].set_title("3. MAE & Uncertainty", fontsize=14)
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Mean Absolute Error", color='tab:blue')
ax3_twin.set_ylabel("Avg Predictive Uncertainty", color='tab:orange')

all_lns3 = lns1 + lns2
labs3 = [l.get_label() for l in all_lns3]
axes[1, 0].legend(all_lns3, labs3, loc='upper right')

# --- Plot 4: RMSE & R² Performance (Bottom Right) ---
ax4_twin = axes[1, 1].twinx()
lns3 = axes[1, 1].plot(rmse_history, marker='D', label='RMSE', color='tab:green', linewidth=2)
lns4 = ax4_twin.plot(r2_history, marker='p', label='$R^2$', color='tab:purple', linewidth=2)

axes[1, 1].set_title("4. RMSE & $R^2$ Performance", fontsize=14)
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("RMSE", color='tab:green')
ax4_twin.set_ylabel("$R^2$ Score", color='tab:purple')
ax4_twin.set_ylim([0, 1.1]) 

all_lns4 = lns3 + lns4
labs4 = [l.get_label() for l in all_lns4]
axes[1, 1].legend(all_lns4, labs4, loc='lower right')

#spaced layout
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(comparison_log['unc_pre'], label='Uncertainty (Pre)', linestyle='--', marker='o')
plt.plot(comparison_log['unc_post'], label='Uncertainty (Post)', linestyle='-', marker='s')
plt.fill_between(range(len(comparison_log['unc_pre'])), 
                 comparison_log['unc_pre'], comparison_log['unc_post'], 
                 color='orange', alpha=0.2, label='Uncertainty Reduction')
plt.title("Knowledge Gain: Uncertainty Evolution")
plt.xlabel("Iteration")
plt.ylabel("Mean Prediction Std Dev")
plt.legend()
plt.show()

In [ ]:
# --- Run this ONCE at the start of your notebook ---
all_samples = [] 

# Add your initial data (The samples you already had before the first iteration)
# Assuming X_train_raw (N, 3) is your historical data
all_samples.append(X_train) 

# --- Run this INSIDE your loop (every time you run 5 new points) ---
# X_new_5 is your (5, 3) array of new volumes
all_samples.append(X_new_5.values)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# --- Original training data ---
fig.add_trace(go.Scatterternary(
    a=X_train["R"],
    b=X_train["Y"],
    c=X_train["B"],
    mode="markers",
    marker=dict(color="black", size=5),
    name="Training Data"
))

# --- Batch 1 (light red) ---
fig.add_trace(go.Scatterternary(
    a=X_new_5["R"],
    b=X_new_5["Y"],
    c=X_new_5["B"],
    mode="markers",
    marker=dict(color="lightcoral", size=8),
    name="Batch 1"
))

# --- Batch 1 Best (dark red) ---
fig.add_trace(go.Scatterternary(
    a=[X_new_5.loc[best_idx_raw, "R"]],
    b=[X_new_5.loc[best_idx_raw, "Y"]],
    c=[X_new_5.loc[best_idx_raw, "B"]],
    mode="markers",
    marker=dict(color="darkred", size=12, symbol="diamond"),
    name="Batch 1 Best"
))



# --- Layout ---
fig.update_layout(
    ternary={
        'sum': 100,
        "aaxis": {
            "title": {"text": "Red", "font": {"color": "red", "size": 20}},
            "tickfont": {"color": "red"},
            "linecolor": "red"
        },
        "baxis": {
            "title": {"text": "Yellow", "font": {"color": "yellow", "size": 20}},
            "tickfont": {"color": "black"},
            "linecolor": "yellow"
        },
        "caxis": {
            "title": {"text": "Blue", "font": {"color": "blue", "size": 20}},
            "tickfont": {"color": "blue"},
            "linecolor": "blue"
        }
    }
)

fig.show()

In [ ]:
def propose_next_batch(model, likelihood, X_pool_scaled, X_pool_raw, X_train_raw, best_y_dist, target_scaled, n_points=5):
    """
    X_train_raw: The history of all volumes you have already tested.
    """
    # 1. Identify which rows in the pool have NOT been tested yet
    # We use a small tolerance (1e-5) in case of floating point math issues
    is_new_mask = []
    for pool_row in X_pool_raw:
        # Check if pool_row is 'near' any row in X_train_raw
        is_repeated = np.any(np.all(np.isclose(X_train_raw, pool_row, atol=1e-5), axis=1))
        is_new_mask.append(not is_repeated)
    
    is_new_mask = np.array(is_new_mask)
    
    # 2. Create a 'Filtered' pool of only new points
    filtered_pool_scaled = X_pool_scaled[is_new_mask]
    filtered_pool_raw = X_pool_raw[is_new_mask]
    
    # 3. Calculate EI only on these new points
    mu_channels, sigma_channels = gp_predict_mean_std(model, likelihood, filtered_pool_scaled)
    mu_dist = np.sum((mu_channels - target_scaled)**2, axis=1)
    sigma_dist = np.mean(sigma_channels, axis=1)
    
    sigma_dist = np.maximum(sigma_dist, 1e-9)
    Z = (best_y_dist - mu_dist) / sigma_dist
    ei = (best_y_dist - mu_dist) * norm.cdf(Z) + sigma_dist * norm.pdf(Z)
    
    # 4. Select top points from the filtered set
    best_indices = np.argsort(ei)[::-1][:n_points]
    
    return filtered_pool_raw[best_indices], filtered_pool_scaled[best_indices], ei[best_indices]

In [ ]:
# --- STEP 1: INPUT YOUR NEW EXPERIMENTAL RESULTS ---
# Replace these with the 5 results you just got back
new_vols_raw = np.array([new_volumes_raw.values])
new_vols_raw = new_vols_raw.reshape(5, 3)

new_resps_raw = np.array([new_responses_raw.values])
new_resps_raw = new_resps_raw.reshape(5, 8)

# --- STEP 2: RECORD "BEFORE" METRICS ---
model.eval()
with torch.no_grad():
    preds_pre = likelihood(model(torch.tensor(X_train_scaled, dtype=torch.float32)))
    mae_pre = np.mean(np.abs(preds_pre.mean.numpy() - y_train_scaled))
    unc_pre = np.mean(preds_pre.stddev.numpy())

# --- STEP 3: UPDATE DATASET & RETRAIN ---
# Combine old data with new data
X_train_raw = np.vstack([X_train, new_vols_raw])
y_train_raw = np.vstack([y_train, new_resps_raw])

# Refit scalers and update tensors
X_train_scaled = x_scaler.fit_transform(X_train_raw)
y_train_scaled = y_scaler.fit_transform(y_train_raw)
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

# Update model and run training loop
model.set_train_data(inputs=X_train_t, targets=y_train_t, strict=False)
model.train(); likelihood.train()
# ... (Insert your Adam optimizer loop here for ~50 iterations) ...

# --- STEP 4: RECORD "AFTER" METRICS ---
model.eval()
with torch.no_grad():
    preds_post = likelihood(model(X_train_t))
    mae_post = np.mean(np.abs(preds_post.mean.numpy() - y_train_scaled))
    unc_post = np.mean(preds_post.stddev.numpy())

# --- STEP 5: PROPOSE THE NEXT 5 POINTS ---
current_dists = np.sum((y_train - target)**2, axis=1)
#current_dists = np.sum((y_train_scaled - target_scaled)**2, axis=1)
best_y_dist = np.min(current_dists)

#next_vols, next_scaled, next_ei = propose_next_batch(model, likelihood, X_pool_raw, best_y_dist, target_scaled, n_points=5)
next_batch_raw, next_batch_scaled, eis = propose_next_batch(model, likelihood, X_pool_scaled, X_pool_raw, X_train_raw, best_y_current, target_scaled, n_points=5)

print(f"Accuracy Improvement: MAE went from {mae_pre:.4f} to {mae_post:.4f}")
print(f"Uncertainty Reduction: Avg Std Dev went from {unc_pre:.4f} to {unc_post:.4f}")
print("\nNext 5 Volume sets to run:")
print(next_batch_raw)

Note: Modifications in the acquisition function were made for channel alignment and metrics calculations. The original second iteration 5 batch samples were (reflected in the analysis below): 

Next 5 Volume sets to run:
[[130.  30. 140.]
 [ 90. 170.  40.]
 [150. 140.  10.]
 [ 60.  10. 230.]
 [ 80. 190.  30.]]

#### Iteration 2: Batch 5 samples (Total 10 Samples)

In [ ]:
new_data_path2 = data_dir / "Rija_10.csv"
new_df_10 = safe_read_csv(new_data_path2, "New data from Rija_10.csv")
new_df_10


In [ ]:
#clean new_df_10 to have only R, Y, B and channels
new_df_10["Command"] = new_df_10["Command"].apply(ast.literal_eval)
new_df_10["Sensor Data"] = new_df_10["Sensor Data"].apply(ast.literal_eval)
cmd_expanded_10 = pd.json_normalize(new_df_10["Command"])
cmd_expanded_10 = cmd_expanded_10.rename(columns={"R": "R", "Y": "Y", "B": "B"})
sensor_expanded_10 = pd.json_normalize(new_df_10["Sensor Data"])
flat_df_10 = pd.concat(
    [
        cmd_expanded_10,          # R, Y, B
        sensor_expanded_10,       # ch583 ... ch440
    ],
    axis=1
)   
channel_cols_10 = [c for c in flat_df_10.columns if c.startswith("ch")]
clean_df_10 = flat_df_10[["R", "Y", "B"] + channel_cols_10].copy()
clean_df_10


In [ ]:
#calculate distance to target for new data
X_new_10 = clean_df_10[["R", "Y", "B"]]
y_new_10 = clean_df_10[[f"{ch}" for ch in channel_cols_10]]    
X_new_10_scaled = x_scaler.transform(X_new_10)

#ordered columns of y_new_10 to match y_train_scaled
#expected_cols = y_scaler.feature_names_in_
y_new_10_aligned = y_new_10[expected_cols]
y_new_10_scaled = y_scaler.transform(y_new_10_aligned)

y_new_10_objective = np.sum((y_new_10_scaled - target_scaled)**2, axis=1)
print(f"Distance to target for new data: {y_new_10_objective}")
#print the closest sample index and its RYB values
closest_idx = np.argmin(y_new_10_objective)
print(f"Closest sample index: {closest_idx}")
print(f"RYB values of closest sample: {X_new_10.iloc[closest_idx].values}")



In [ ]:
# --- 1. Calculate Distances ---

# Iteration 0: The best (minimum) distance in your original training data
initial_distances = np.sum((y_train_scaled[:-(len(y_new_5_scaled))] - target_scaled)**2, axis=1)
best_dist_it0 = np.min(initial_distances)

# Iteration 1: The distances of the 5 new points you just added
# We use the mean distance of the batch to show the average quality of the proposal
new_points_distances = np.sum((y_new_5_scaled - target_scaled)**2, axis=1)
avg_dist_it1 = np.mean(new_points_distances)
best_dist_it1 = np.min(new_points_distances)

# Store for plotting
iterations = [0, 1]
best_distances = [best_dist_it0, best_dist_it1]
batch_avg_distances = [best_dist_it0, avg_dist_it1] # Start it0 at best for comparison

In [ ]:
# 1. Calculate Distances for plotting history

# Construct the lists for plotting from available variables:
# best_dist_it0, best_dist_it1, avg_dist_it1 are from cell 99
# y_new_10_objective is from cell 98

plot_best_dists = [best_dist_it0, best_dist_it1, np.min(y_new_10_objective)]
plot_avg_dists = [best_dist_it0, avg_dist_it1, np.mean(y_new_10_objective)] # Using best_dist_it0 as a placeholder for iteration 0 average
plot_iters = range(len(plot_best_dists))

# 2. Dynamic Plotting
plt.figure(figsize=(10, 6))

# Target baseline
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Perfect Match (J=0)')

# Best Distance Trend (The main convergence line)
plt.plot(plot_iters, plot_best_dists, marker='o', markersize=8, color='tab:blue',
         linewidth=3, label='Best Sample in Batch')

# Avg Distance Scatter (Shows the "exploration" spread of the batch)
plt.scatter(plot_iters, plot_avg_dists, color='tab:orange', s=100,
            label='Avg. Batch Distance', zorder=5, marker='x')

plt.fill_between(plot_iters, 0, plot_best_dists, color='green', alpha=0.1)

plt.title("Convergence: Evolution of Spectral Distance", fontsize=14)
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Squared Euclidean Distance $J(x)$", fontsize=12)
plt.xticks(plot_iters) # Automatically sets ticks based on iteration count
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. 'PRE' Metrics (Current model on existing data) ---
model.eval()
likelihood.eval()
with torch.no_grad():
    # Make predictions on the current full training set (which includes Iteration 1's new data)
    preds_pre_iter2 = likelihood(model(X_train_t))
    mean_preds_pre_iter2 = preds_pre_iter2.mean.numpy()
    std_preds_pre_iter2 = preds_pre_iter2.stddev.numpy()

    # Calculate MAE for 'pre' metrics
    mae_pre_iter2 = mean_absolute_error(y_train_scaled, mean_preds_pre_iter2)
    comparison_log['mae_pre'].append(mae_pre_iter2)

    # Calculate mean uncertainty for 'pre' metrics
    unc_pre_iter2 = np.mean(std_preds_pre_iter2)
    comparison_log['unc_pre'].append(unc_pre_iter2)

    # Calculate minimum distance to target for 'pre' metrics
    # We need to compute objective values for the current scaled training data
    current_y_train_objective = np.sum((y_train_scaled - target_scaled)**2, axis=1)
    dist_pre_iter2 = np.min(current_y_train_objective)
    comparison_log['dist_pre'].append(dist_pre_iter2)

print(f"Pre-update MAE: {mae_pre_iter2:.4f}")
print(f"Pre-update Uncertainty: {unc_pre_iter2:.4f}")
print(f"Pre-update Min Distance: {dist_pre_iter2:.4f}")

In [ ]:
# --- 2. STEP A: INPUT NEW DATA (Iteration 2 batch) ---
# make dataframe new_volumes_raw_iter2 with columns R, Y, B from clean_df_10
new_volumes_raw_iter2 = clean_df_10[["R", "Y", "B"]].copy()

# make dataframe new_responses_raw_iter2 with channel columns from clean_df_10
new_responses_raw_iter2 = clean_df_10[[f"{ch}" for ch in channel_cols_10]].copy()
# make sure the columns of new_responses_raw_iter2 are in the same order as y_train
new_responses_raw_iter2 = new_responses_raw_iter2[y_train.columns]

# --- 3. STEP B: UPDATE TRAINING SET ---
X_train = pd.concat([X_train, new_volumes_raw_iter2], ignore_index=True)
y_train = pd.concat([y_train, new_responses_raw_iter2], ignore_index=True)

# --- 4. STEP C: REFIT SCALERS AND UPDATE MODEL ---
# Refit on updated training data ONLY
x_scaler.fit(X_train)
y_scaler.fit(y_train)

X_train_scaled = x_scaler.transform(X_train)
y_train_scaled = y_scaler.transform(y_train)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

# Update GPyTorch model with new data
model.set_train_data(inputs=X_train_t, targets=y_train_t, strict=False)

# Retrain model (using parameters similar to cell 64 for thoroughness)
model.train()
likelihood.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
num_iters = 1000 # Increased iterations for better convergence
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

print("Starting GPyTorch model retraining for Iteration 2...")
for i in range(num_iters):
    optimizer.zero_grad()
    with gpytorch.settings.cholesky_jitter(1e-3):
        output = model(X_train_t)
        loss = -mll(output, y_train_t)
    loss.backward()
    optimizer.step()
    if i % 200 == 0: # Reduced print frequency for brevity
        print(f"Iter {i} (Iteration 2) - Loss: {loss.item():.4f}")
print("GPyTorch model retraining complete for Iteration 2.")

# STEP D: Calculate and append post-update performance metrics for Iteration 2
model.eval()
likelihood.eval()
with torch.no_grad():
    preds_post_iter2 = likelihood(model(X_train_t))
    mean_preds_on_full_train_set = preds_post_iter2.mean.numpy()
    std_preds_on_full_train_set = torch.sqrt(preds_post_iter2.variance).numpy()

# Convergence: Average squared distance of predicted mean to target_scaled (over the entire updated training set)
current_dist = np.mean(np.sum((mean_preds_on_full_train_set - target_scaled)**2, axis=1))
dist_to_target_history.append(current_dist)

# Accuracy metrics on training set
current_mae = mean_absolute_error(y_train_scaled, mean_preds_on_full_train_set)
current_rmse = root_mean_squared_error(y_train_scaled, mean_preds_on_full_train_set)
current_r2 = r2_score(y_train_scaled, mean_preds_on_full_train_set)
mae_history.append(current_mae)
rmse_history.append(current_rmse)
r2_history.append(current_r2)

# Uncertainty: Average standard deviation over the entire updated training set
current_unc = np.mean(std_preds_on_full_train_set)
uncertainty_history.append(current_unc)

print(f"Iteration 2 Complete. Latest Average Predicted Distance (full training set): {current_dist:.4f}")

# STEP E: Append post-update metrics to `comparison_log`
comparison_log['mae_post'].append(current_mae)
comparison_log['unc_post'].append(current_unc)
comparison_log['dist_post'].append(np.min(np.sum((mean_preds_on_full_train_set - target_scaled)**2, axis=1)))

# STEP F: Update `all_samples` for 3D plot
all_samples.append(new_volumes_raw_iter2.values)

In [ ]:
# --- Prepare for Latest Batch Spectral Profile Plot ---
# Reassign X_new_5 and y_new_5 to refer to the data from the second new batch
#X_new_5 = X_new_10.copy()
#y_new_5 = y_new_10.copy()

# --- Generate Plots ---
# Generate Convergence Plot (Cell 100)
plot_best_dists = [best_dist_it0, best_dist_it1, np.min(y_new_10_objective)]
plot_avg_dists = [best_dist_it0, avg_dist_it1, np.mean(y_new_10_objective)]
plot_iters = range(len(plot_best_dists))

plt.figure(figsize=(10, 6))
plt.axhline(0, color='black', linestyle='--', linewidth=1.5, label='Perfect Match (J=0)')
plt.plot(plot_iters, plot_best_dists, marker='o', markersize=8, color='tab:blue',
         linewidth=3, label='Best Sample in Batch')
plt.scatter(plot_iters, plot_avg_dists, color='tab:orange', s=100,
            label='Avg. Batch Distance', zorder=5, marker='x')
plt.fill_between(plot_iters, 0, plot_best_dists, color='green', alpha=0.1)
plt.title("Convergence: Evolution of Spectral Distance", fontsize=14)
plt.xlabel("Iteration", fontsize=12)
plt.ylabel("Squared Euclidean Distance $J(x)$", fontsize=12)
plt.xticks(plot_iters)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Generate Spectral Profile Plot for Latest Batch (Cell 83)
original_channel_names = y_train.columns.tolist()
target_df = pd.Series(target_raw_values, index=original_channel_names)
target_aligned = target_df.reindex(y_new_10.columns).values
target_raw_plot = target_aligned

raw_responses = y_new_10.values
raw_distances = np.sum((raw_responses - target_raw_plot)**2, axis=1)
best_idx_raw = np.argmin(raw_distances)

colors = ['tab:blue', 'tab:orange', 'tab:purple', 'tab:red', 'tab:brown']
channels = y_new_10.columns.tolist()

plt.figure(figsize=(12, 7))
plt.plot(channels, target_raw_plot, 'k--', marker='s', label='TARGET (Ideal)', linewidth=3, zorder=10)

for i in range(len(raw_responses)):
    current_raw = raw_responses[i]
    current_color = colors[i % len(colors)]

    if i == best_idx_raw:
        plt.plot(channels, current_raw, color='tab:green', marker='o',
                 linewidth=4, label=f'SAMPLE {i} (BEST MATCH)', zorder=5)
        plt.fill_between(channels, target_raw_plot, current_raw, color='tab:green', alpha=0.1)
    else:
        plt.plot(channels, current_raw, color=current_color, alpha=0.4,
                 linewidth=1.5, marker='.', label=f'Sample {i}')

plt.title("Spectral Profile: Comparison of Second Set of 5 Batch Samples (Raw)", fontsize=16)
plt.ylabel("Raw Sensor Intensity (Counts)", fontsize=12)
plt.xlabel("Wavelength Channel", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10, frameon=True)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()
print("Volumes sampled in this batch:")
print(X_new_10)

In [ ]:
# Generate Accuracy and Uncertainty Evolution Plots (Cell 88)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(dist_to_target_history, marker='o', color='tab:red', linewidth=2, label='Current Best')
axes[0, 0].axhline(0, color='black', linestyle='--', linewidth=2, label='Target (0)')
axes[0, 0].set_title("1. Convergence (with Target)", fontsize=14)
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Distance $J(x)$")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(dist_to_target_history, marker='o', color='tab:red', linewidth=2)
axes[0, 1].set_title("2. Convergence (Zoomed)", fontsize=14)
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("Distance $J(x)$")
axes[0, 1].grid(True, alpha=0.3)

ax3_twin = axes[1, 0].twinx()
lns1 = axes[1, 0].plot(mae_history, marker='s', label='MAE', color='tab:blue', linewidth=2)
lns2 = ax3_twin.plot(uncertainty_history, marker='^', label='Uncertainty ($\sigma$)', color='tab:orange', alpha=0.7)

axes[1, 0].set_title("3. MAE & Uncertainty", fontsize=14)
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Mean Absolute Error", color='tab:blue')
ax3_twin.set_ylabel("Avg Predictive Uncertainty", color='tab:orange')

all_lns3 = lns1 + lns2
labs3 = [l.get_label() for l in all_lns3]
axes[1, 0].legend(all_lns3, labs3, loc='upper right')

ax4_twin = axes[1, 1].twinx()
lns3 = axes[1, 1].plot(rmse_history, marker='D', label='RMSE', color='tab:green', linewidth=2)
lns4 = ax4_twin.plot(r2_history, marker='p', label='$R^2$', color='tab:purple', linewidth=2)

axes[1, 1].set_title("4. RMSE & $R^2$ Performance", fontsize=14)
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("RMSE", color='tab:green')
ax4_twin.set_ylabel("$R^2$ Score", color='tab:purple')
ax4_twin.set_ylim([0, 1.1])

all_lns4 = lns3 + lns4
labs4 = [l.get_label() for l in all_lns4]
axes[1, 1].legend(all_lns4, labs4, loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# Generate Uncertainty Comparison Plot (Cell 89)
plt.figure(figsize=(8, 5))
# Determine the number of valid entries to plot to avoid ValueError
num_entries_to_plot = min(len(comparison_log['unc_pre']), len(comparison_log['unc_post']))

plt.plot(comparison_log['unc_pre'][:num_entries_to_plot], label='Uncertainty (Pre)', linestyle='--', marker='o')
plt.plot(comparison_log['unc_post'][:num_entries_to_plot], label='Uncertainty (Post)', linestyle='-', marker='s')
plt.fill_between(range(num_entries_to_plot),
                 comparison_log['unc_pre'][:num_entries_to_plot], comparison_log['unc_post'][:num_entries_to_plot],
                 color='orange', alpha=0.2, label='Uncertainty Reduction')
plt.title("Knowledge Gain: Uncertainty Evolution")
plt.xlabel("Iteration")
plt.ylabel("Mean Prediction Std Dev")
plt.legend()
plt.show()

In [ ]:
#ternary plot 
fig = px.scatter_ternary(
    X_train, 
    a="R", 
    b="Y", 
    c="B",
)

fig.update_traces(marker=dict(color='black', size=5))

fig.update_layout(
    ternary={
        'sum': 100,
        "aaxis": {
            "title": {"text": "Red", "font": {"color": "red", "size": 20}},
            "tickfont": {"color": "red"},
            "linecolor": "red"
        },
        "baxis": {
            "title": {"text": "Yellow", "font": {"color": "yellow", "size": 20}},
            "tickfont": {"color": "black"},
            "linecolor": "yellow"
        },
        "caxis": {
            "title": {"text": "Blue", "font": {"color": "blue", "size": 20}},
            "tickfont": {"color": "blue"},
            "linecolor": "blue"
        }
    }
)
fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# --- Original training data ---
fig.add_trace(go.Scatterternary(
    a=X_train["R"],
    b=X_train["Y"],
    c=X_train["B"],
    mode="markers",
    marker=dict(color="black", size=5),
    name="Training Data"
))

# --- Batch 1 (light red) ---
fig.add_trace(go.Scatterternary(
    a=X_new_5["R"],
    b=X_new_5["Y"],
    c=X_new_5["B"],
    mode="markers",
    marker=dict(color="lightcoral", size=8),
    name="Batch 1"
))

# --- Batch 1 Best (dark red) ---
fig.add_trace(go.Scatterternary(
    a=[X_new_5.loc[best_idx_raw, "R"]],
    b=[X_new_5.loc[best_idx_raw, "Y"]],
    c=[X_new_5.loc[best_idx_raw, "B"]],
    mode="markers",
    marker=dict(color="darkred", size=12, symbol="diamond"),
    name="Batch 1 Best"
))

# --- Batch 2 (light blue) ---
fig.add_trace(go.Scatterternary(
    a=X_new_10["R"],
    b=X_new_10["Y"],
    c=X_new_10["B"],
    mode="markers",
    marker=dict(color="lightblue", size=8),
    name="Batch 2"
))

# --- Batch 2 Best (dark blue) ---
fig.add_trace(go.Scatterternary(
    a=[X_new_10.loc[closest_idx, "R"]],
    b=[X_new_10.loc[closest_idx, "Y"]],
    c=[X_new_10.loc[closest_idx, "B"]],
    mode="markers",
    marker=dict(color="darkblue", size=12, symbol="diamond"),
    name="Batch 2 Best"
))

# --- Layout ---
fig.update_layout(
    ternary={
        'sum': 100,
        "aaxis": {
            "title": {"text": "Red", "font": {"color": "red", "size": 20}},
            "tickfont": {"color": "red"},
            "linecolor": "red"
        },
        "baxis": {
            "title": {"text": "Yellow", "font": {"color": "yellow", "size": 20}},
            "tickfont": {"color": "black"},
            "linecolor": "yellow"
        },
        "caxis": {
            "title": {"text": "Blue", "font": {"color": "blue", "size": 20}},
            "tickfont": {"color": "blue"},
            "linecolor": "blue"
        }
    }
)

fig.show()

## Summary 

### Surrogate Model Performance
- The final active learning process employed a Gaussian Process Regression (GPR) surrogate model, specifically the MultitaskSMGPModel implemented using GPyTorch with a SpectralMixtureKernel. 
- This model was trained on initial students_data set and was updated later on with the acquistion policy.
- After initial training, the model's performance metrics were recorded, showing reasonable accuracy on the training data (mae_train ~0.23, rmse_train ~0.37, r2_train ~0.86 for GPyTorch_mod2).
- Although validation metrics were lower, the active learning loop focuses on improving performance on the expanding training set with strategically selected values using the acquisition policy. The GPR model's strength lies in its ability to provide uncertainty estimates, which are crucial for guiding the active learning process.

### Acquisition Policy: Expected Improvement (Exploitation/Exploration)
- The acquisition policy utilized is Expected Improvement (EI). This policy strategically balances exploitation (sampling points near the current best known optimum) and exploration (sampling in regions where the model's uncertainty is high). The mechanism works as follows:
    -  The surrogate model predicts the mean (mu_channels) and standard deviation (sigma_channels) for potential candidate points across all 8 channels.
    - These multi-channel predictions are converted into a scalar objective function (mu_dist) representing the squared Euclidean distance to the normalized target response, and an aggregate uncertainty (sigma_dist) for this objective.
    - EI then calculates a score for each candidate point by considering both how much improvement is expected (exploitation) and the probability of finding a significantly better point in uncertain regions (exploration). A higher EI value indicates a more promising point to sample next.
    - The propose_next_batch function selects the n_points (here, 5) with the highest EI values from the candidate pool, ensuring that already sampled points are not re-selected. This iterative process allows the model to efficiently converge towards the optimal RYB mixture while also expanding its knowledge base in less understood areas of the design space.

### Active Learning Progress: Iteration 1 vs. Iteration 2

 - Convergence: Evolution of Spectral Distance: 
 This plot demonstrates a clear trend of decreasing spectral distance ( 𝐽(𝑥) ) from the initial state (Iteration 0).  The 'Best Sample in Batch' value, which tracks the minimum distance found in each newly sampled batch, showed a significant improvement from Iteration 0's best (0.118) to Iteration 1's best (1.462, though the best sample from the entire updated dataset was 0.1135, indicating a strong early find) however it further increased for batch 2's best point in Iteration 2. The overall best point was index 3 from batch 1. Both iterations however produced a best point that was significantly better than the average indicating that the acquisiton policy is moving in the right direction.

- Spectral Profile Comparison: These plots illustrate how the spectral response of the best samples in each new batch compares to the target.
    - Iteration 1: Sample 3 with (RYB) = (30, 210, 60) and had raw distance of 1.462 and visually showed a reasonable alignment with the target profile across most channels.
    - Iteration 2: Sample 2 with (RYB) = (150, 140, 10) and had a raw distance of 4.193, indicating that although this is the best sample in the second batch the first best is the best overall. 

### Accuracy & Uncertainty Evolution: These plots track the surrogate model's performance on the continually expanding training dataset.

- MAE & Uncertainty: As one would expect, the Mean Absolute Error (MAE) and the average predictive uncertainty (𝜎) also increased but very slightly, which is indicates that although the best from the second batch performed worse than the first, it is not by that much. 

- RMSE & R² Performance: Similar to MAE, Root Mean Squared Error (RMSE) slightly, while the R² score consistently remained high  reaffirming the model's the better performance by the best in batch 1. 

- Knowledge Gain: Uncertainty Evolution: Accordingly the uncertainty also increased from the bests in batch 1 and 2 highlighted by the orange area. 

### Ternary Convergence: This plot visualizes the evolution of sampled RYB points.

- It shows the initial distribution of the training data and then how the batches from Iteration 1 and Iteration 2 were added. The plot indicates that the active learning strategy, driven by EI, is effectively exploring the RYB space.
- We can see that the two best point are localized in a similar region in the RYB space. This indicates that the surrogate model and acquisition function are effectively able to narrow down into the area of the RYB space where the target resides. 

In conclusion, the active learning process has demonstrably improved the surrogate model's predictive accuracy and significantly reduced its uncertainty over the first two iterations. The Expected Improvement acquisition policy effectively balances exploration and exploitation, leading to a focused and efficient search for the optimal RYB mixture. The visual analysis of the plots confirms that the active learning strategy is successfully driving the model closer to the target response.

# “Does active learning make the data more “informative”, or are you just collecting more data?

The active learning was quite effective in the first batch of 5 samples. Given the nature of this task, those 5 samples were suggested at the same time. It is likely that the acquisiton function would not need that many tries to narrow in on exploiting the target region. Given this scenario where the goal was to explicitly depict the target (exploitation) more data was not required. 

This is however explicit to this task. This is a subjective question that would vary from the goal of different task. As seen in the ternary plot above, there is a limited coverage of the RYB space. If the goal was RYB exploration and a sparser training set was used, more data would definitely be required. 